# CMV JSONL to Manual Annotation CSVs

This notebook reads the local CMV JSONL dumps under `cmw/`, picks a post with at least `300` comments, and writes the three CSV files used elsewhere in this repo for manual annotation:

- `comments.csv`
- `nodes.csv`
- `annotation_template.csv`

By default it selects the qualifying post with the highest `num_comments`, but you can override that with `TARGET_POST_ID`.

In [1]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path("cmw")
POSTS_PATH = DATA_DIR / "r_changemyview_posts.jsonl"
COMMENTS_PATH = DATA_DIR / "r_changemyview_comments.jsonl"

MIN_COMMENTS = 300
TARGET_POST_ID: str | None = None
OUT_ROOT = Path("generated_threads")

for p in (POSTS_PATH, COMMENTS_PATH):
    if not p.exists():
        raise FileNotFoundError(p)

OUT_ROOT.mkdir(exist_ok=True)


In [2]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def normalize_author(author):
    if author is None:
        return None
    author = str(author).strip()
    if not author or author.lower() in {"[deleted]", "deleted", "none"}:
        return None
    return author


def slugify(text: str, max_len: int = 60) -> str:
    text = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
    return text[:max_len].rstrip("_") or "untitled"


def choose_post(posts_df: pd.DataFrame, min_comments: int, target_post_id: str | None = None) -> pd.Series:
    qualifying = posts_df[posts_df["num_comments"].fillna(0).astype(int) >= min_comments].copy()
    if qualifying.empty:
        raise ValueError(f"No posts found with at least {min_comments} comments.")

    qualifying = qualifying.sort_values(["num_comments", "created_utc"], ascending=[False, True])
    if target_post_id is None:
        return qualifying.iloc[0]

    match = qualifying[qualifying["id"].astype(str) == str(target_post_id)]
    if match.empty:
        raise ValueError(
            f"Post id {target_post_id!r} is missing or has fewer than {min_comments} comments."
        )
    return match.iloc[0]


def build_comments_df(post_row: pd.Series, comments_df: pd.DataFrame) -> pd.DataFrame:
    post_id = str(post_row["id"])
    post_fullname = f"t3_{post_id}"

    thread = comments_df[comments_df["link_id"].astype(str) == post_fullname].copy()
    if thread.empty:
        raise ValueError(f"No comments found for post {post_id}.")

    thread["author"] = thread["author"].map(normalize_author)
    thread = thread.dropna(subset=["author", "id"]).copy()
    thread["id"] = thread["id"].astype(str)
    thread["parent_id"] = thread["parent_id"].astype(str)

    id_to_author = dict(zip(thread["id"], thread["author"]))
    id_to_parent = dict(zip(thread["id"], thread["parent_id"]))
    post_author = normalize_author(post_row.get("author"))

    def parent_comment_id(parent_id: str):
        return parent_id[3:] if isinstance(parent_id, str) and parent_id.startswith("t1_") else None

    def parent_author(parent_id: str):
        if not isinstance(parent_id, str):
            return None
        if parent_id.startswith("t1_"):
            return id_to_author.get(parent_id[3:])
        if parent_id == post_fullname:
            return post_author
        return None

    depth_cache: dict[str, int] = {}

    def depth_for(comment_id: str) -> int:
        if comment_id in depth_cache:
            return depth_cache[comment_id]

        seen = set()
        cur = comment_id
        depth = 0
        while True:
            parent = id_to_parent.get(cur)
            if not isinstance(parent, str):
                break
            if parent == post_fullname:
                break
            if not parent.startswith("t1_"):
                break
            parent_comment = parent[3:]
            if parent_comment in seen:
                break
            seen.add(parent_comment)
            depth += 1
            cur = parent_comment

        depth_cache[comment_id] = depth
        return depth

    out = pd.DataFrame(
        {
            "submission_id": post_id,
            "submission_title": post_row.get("title"),
            "subreddit": post_row.get("subreddit"),
            "comment_id": thread["id"],
            "author": thread["author"],
            "parent_comment_id": thread["parent_id"].map(parent_comment_id),
            "body": thread["body"].fillna(""),
            "score": thread.get("score"),
            "created_utc": thread.get("created_utc"),
            "permalink": thread["permalink"].map(
                lambda p: f"https://www.reddit.com{p}" if isinstance(p, str) and p else None
            ),
            "depth": thread["id"].map(depth_for),
        }
    )
    out["parent_author"] = thread["parent_id"].map(parent_author)
    return out.sort_values(["created_utc", "comment_id"], na_position="last").reset_index(drop=True)


def build_nodes_df(comments_out: pd.DataFrame) -> pd.DataFrame:
    grouped = defaultdict(list)
    for _, row in comments_out.iterrows():
        grouped[row["author"]].append(str(row["body"]))

    node_rows = [
        {
            "user_id": user_id,
            "num_comments": len(texts),
            "all_text": "\n\n".join(texts).strip(),
        }
        for user_id, texts in grouped.items()
    ]
    nodes_df = pd.DataFrame(node_rows)
    return nodes_df.sort_values(["num_comments", "user_id"], ascending=[False, True]).reset_index(drop=True)


def build_annotation_template(nodes_df: pd.DataFrame) -> pd.DataFrame:
    ann_df = pd.DataFrame({"user_id": nodes_df["user_id"], "stance": "", "notes": ""})
    return ann_df.sort_values("user_id").reset_index(drop=True)


In [3]:
posts_df = load_jsonl(POSTS_PATH)
comments_df = load_jsonl(COMMENTS_PATH)

qualifying_posts = (
    posts_df.loc[posts_df["num_comments"].fillna(0).astype(int) >= MIN_COMMENTS, ["id", "author", "title", "num_comments", "created_utc", "permalink"]]
    .sort_values(["num_comments", "created_utc"], ascending=[False, True])
    .reset_index(drop=True)
)

display(qualifying_posts)
print(f"Qualifying posts: {len(qualifying_posts)}")


,id,author,title,num_comments,created_utc,permalink
0,11f3wmk,hammocknap5,"CMV: ""Cancel culture"" is a just boogieman for ...",900,1677677650,/r/changemyview/comments/11f3wmk/cmv_cancel_cu...
1,11fagvc,eterevsky,"CMV: Despite numerous flaws and controversies,...",701,1677693511,/r/changemyview/comments/11fagvc/cmv_despite_n...
2,11fkz9f,[deleted],CMV: circumcision of infants and baby's should...,683,1677711329,/r/changemyview/comments/11fkz9f/cmv_circumcis...
3,11f4f06,jsn12620,CMV: If you identify as male then you should h...,332,1677679032,/r/changemyview/comments/11f4f06/cmv_if_you_id...


Qualifying posts: 4


In [4]:
selected_post = choose_post(posts_df, min_comments=MIN_COMMENTS, target_post_id=TARGET_POST_ID)
selected_post_id = str(selected_post["id"])
selected_slug = slugify(str(selected_post["title"]))
out_dir = OUT_ROOT / f"reddit_cmv_{selected_post_id}_{selected_slug}"
out_dir.mkdir(parents=True, exist_ok=True)

print("Selected post:")
print(f"  id: {selected_post_id}")
print(f"  title: {selected_post['title']}")
print(f"  num_comments metadata: {selected_post['num_comments']}")
print(f"  out_dir: {out_dir}")


Selected post:
  id: 11f3wmk
  title: CMV: "Cancel culture" is a just boogieman for "letting the free market of humanity decide somebody is an a**hole".
  num_comments metadata: 900
  out_dir: generated_threads/reddit_cmv_11f3wmk_cmv_cancel_culture_is_a_just_boogieman_for_letting_the_free


In [5]:
comments_out = build_comments_df(selected_post, comments_df)
nodes_out = build_nodes_df(comments_out)
annotation_out = build_annotation_template(nodes_out)

print(f"Extracted comments: {len(comments_out)}")
print(f"Users / nodes: {len(nodes_out)}")

display(comments_out.head(3))
display(nodes_out.head(10))
display(annotation_out.head(10))


Extracted comments: 793
Users / nodes: 265


,submission_id,submission_title,subreddit,comment_id,author,parent_comment_id,body,score,created_utc,permalink,depth,parent_author
0,11f3wmk,"CMV: ""Cancel culture"" is a just boogieman for ...",changemyview,jahfgoq,stewshi,jaheuqq,You have the prime time to give an example of ...,0,1677678121,https://www.reddit.com/r/changemyview/comments...,1,None
1,11f3wmk,"CMV: ""Cancel culture"" is a just boogieman for ...",changemyview,jahgdbe,hammocknap5,jahg8g7,Bingo. A correct point I didn't directly hit b...,-2,1677678560,https://www.reddit.com/r/changemyview/comments...,1,None
2,11f3wmk,"CMV: ""Cancel culture"" is a just boogieman for ...",changemyview,jahgwj9,KpYugai,jahfgoq,Not OC but pretty sure they meant\n\nIf Republ...,0,1677678812,https://www.reddit.com/r/changemyview/comments...,2,stewshi


,user_id,num_comments,all_text
0,Phyltre,33,"Meta-engagement with a system (aka ""gaming a s..."
1,Absenceofavoid,23,How do you boycott a person who is an open rac...
2,jweezy2045,23,"Harassment is a crime, and cancel culture is n..."
3,ZappSmithBrannigan,21,"What specifically entails ""cancelling""? \n\nWh..."
4,pawnman99,17,"I love this accusation, because it works every..."
5,Long-Rate-445,16,this is literally exactly the paradox of intol...
6,Terrible_Lift,16,To be fair I’m glad Twitter did that with the ...
7,MeanderingDuck,15,A free market is one where one can buy and sel...
8,BigPickleFucker,13,Lynching is appropriate. Just because it's ass...
9,H0w-1nt3r3st1ng,12,PART 1: I'm a pluralist who is heavily Left le...


,user_id,stance,notes
0,1block,,
1,24Willard,,
2,3720-To-One,,
3,4ku2,,
4,7in7turtles,,
5,AbolishDisney,,
6,Absenceofavoid,,
7,AcerbicCapsule,,
8,Adam-West,,
9,Admirable_Ad1947,,


In [6]:
comments_path = out_dir / "comments.csv"
nodes_path = out_dir / "nodes.csv"
annotation_path = out_dir / "annotation_template.csv"

comments_out.to_csv(comments_path, index=False)
nodes_out.to_csv(nodes_path, index=False)
annotation_out.to_csv(annotation_path, index=False)

print(f"Wrote {comments_path}")
print(f"Wrote {nodes_path}")
print(f"Wrote {annotation_path}")


Wrote generated_threads/reddit_cmv_11f3wmk_cmv_cancel_culture_is_a_just_boogieman_for_letting_the_free/comments.csv
Wrote generated_threads/reddit_cmv_11f3wmk_cmv_cancel_culture_is_a_just_boogieman_for_letting_the_free/nodes.csv
Wrote generated_threads/reddit_cmv_11f3wmk_cmv_cancel_culture_is_a_just_boogieman_for_letting_the_free/annotation_template.csv
